In [2]:
import os
import cv2
import json
import torch
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from glob import glob as gg
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
)

In [5]:
images_path = "symbol_images"

In [8]:
images_count = len(gg(f"symbol_images/*/*"))

images_count

701773

In [9]:
# data = "/content/pre_training"

# data = "/content/drive/MyDrive/Datasets/pre_training"

dataset = tf.keras.utils.image_dataset_from_directory(
    images_path, image_size = (64, 64), batch_size = 64
)

Found 701773 files belonging to 53 classes.


In [10]:
def image_processing(image, label):

    image = tf.cast(image / 255.0, tf.float32)

    return image, label

In [12]:
processed_data = dataset.map(image_processing)

processed_data = processed_data.shuffle(buffer_size = 1000, seed = 42)

In [13]:
train_size = int(len(processed_data) * 0.6)
test_size = int(len(processed_data) * 0.2)
val_size = int(len(processed_data) * 0.2) + 1

assert train_size + test_size + val_size == len(processed_data)

len(processed_data), train_size, test_size, val_size

(10966, 6579, 2193, 2194)

In [14]:
train = processed_data.take(train_size)
test = processed_data.skip(train_size).take(test_size)
val = processed_data.skip(train_size + test_size).take(val_size)

In [15]:
assert len(train) + len(test) + len(val) == len(processed_data)

In [16]:
train = train.prefetch(buffer_size = tf.data.AUTOTUNE)
test = test.prefetch(buffer_size = tf.data.AUTOTUNE)
val = val.prefetch(buffer_size = tf.data.AUTOTUNE)

In [17]:
model = Sequential([
    Conv2D(
        filters = 32, kernel_size = (3, 3),
        activation = 'relu', input_shape = (64, 64, 3)
    ),
    BatchNormalization(),
    MaxPooling2D(pool_size = (3, 3)), # Imshape = 56, 56
    Conv2D(
        filters = 64, kernel_size = (2, 2),
        activation = 'relu'
    ),
    BatchNormalization(),
    MaxPooling2D(pool_size = (2, 2)),
    Conv2D(
        filters = 128, kernel_size = (2, 2),
        activation = 'relu'
    ),
    BatchNormalization(),
    MaxPooling2D(pool_size = (2, 2)),
    Conv2D(
        filters = 64, kernel_size = (2, 2),
        activation = 'relu'
    ),
    BatchNormalization(),
    MaxPooling2D(pool_size = (2, 2)),
    Flatten(),
    Dense(units = 32, activation = 'relu'),
    Dropout(rate = 0.2),
    Dense(units = 53, activation = 'softmax')
])

/Users/admin/Downloads/Code/SCHOOL_PROJECT/.env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [20]:
model_count = len(gg('models/*.keras'))

best_weight_cb = ModelCheckpoint(
    filepath = f"models/symbol_model_{model_count + 1}.keras",
    monitor = "val_accuracy",
    verbose = 1,
    save_best_only = True
)

model.compile(
    # optimizer = Adam(learning_rate = lr_scheduler),
    optimizer = Adam(learning_rate = 0.001),
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)

# early_stopping_cb = EarlyStopping(
#     monitor = "val_loss",
#     patience = 3,
#     restore_best_weights = True
# )

# lr_scheduler = ExponentialDecay(
#     initial_learning_rate = 0.001,
#     decay_steps = 10000,
#     decay_rate = 0.96,
#     staircase = True
# )

In [21]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 62, 62, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 20, 20, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 19, 19, 64)     │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 19, 19, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 9, 9, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 3, 3, 64)       │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 3, 3, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 1, 1, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 53)             │         1,749 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,861 (311.96 KB)

 Trainable params: 79,285 (309.71 KB)

 Non-trainable params: 576 (2.25 KB)

In [22]:
for images, labels in train.take(1):
    preds = model.predict(images)
    print("Predictions shape:", preds.shape)
    print("Labels shape:", labels.shape)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step 
Predictions shape: (64, 53)
Labels shape: (64,)


2025-04-12 11:34:27.649840: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [ ]:
model = model.fit(
    train,
    validation_data = val,
    batch_size = 64,
    epochs = 50,
    verbose = 1,
    callbacks = [best_weight_cb]
)

Epoch 1/50
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.6382 - loss: 1.3305
Epoch 1: val_accuracy improved from -inf to 0.77490, saving model to models/symbol_model_2.keras
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 724s 109ms/step - accuracy: 0.6382 - loss: 1.3304 - val_accuracy: 0.7749 - val_loss: 0.7310
Epoch 2/50
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.8201 - loss: 0.5765

2025-04-12 12:00:24.785732: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:452] ShuffleDatasetV3:10: Filling up shuffle buffer (this may take a while): 853 of 1000
2025-04-12 12:00:26.822623: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:482] Shuffle buffer filled.



Epoch 2: val_accuracy did not improve from 0.77490
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 976s 147ms/step - accuracy: 0.8201 - loss: 0.5765 - val_accuracy: 0.6157 - val_loss: 2.0262
Epoch 3/50


2025-04-12 12:03:15.219596: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:452] ShuffleDatasetV3:10: Filling up shuffle buffer (this may take a while): 893 of 1000
2025-04-12 12:03:16.531635: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:482] Shuffle buffer filled.


6579/6579 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.8437 - loss: 0.4930

...

6579/6579 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - accuracy: 0.9103 - loss: 0.2626
Epoch 35: val_accuracy did not improve from 0.91296
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 827s 124ms/step - accuracy: 0.9103 - loss: 0.2626 - val_accuracy: 0.8982 - val_loss: 0.3140
Epoch 36/50
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.9108 - loss: 0.2607
Epoch 36: val_accuracy did not improve from 0.91296
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 696s 104ms/step - accuracy: 0.9108 - loss: 0.2607 - val_accuracy: 0.8959 - val_loss: 0.3278
Epoch 37/50
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.9120 - loss: 0.2575
Epoch 37: val_accuracy did not improve from 0.91296
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 734s 110ms/step - accuracy: 0.9120 - loss: 0.2575 - val_accuracy: 0.8032 - val_loss: 0.8704
Epoch 38/50
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.9125 - loss: 0.2550
Epoch 38: val_accuracy did not improve from 0.91296
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 716s 108ms/step - accuracy: 0.9125 - loss: 

2025-04-12 21:34:26.457967: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:452] ShuffleDatasetV3:10: Filling up shuffle buffer (this may take a while): 902 of 1000


   1/6579 ━━━━━━━━━━━━━━━━━━━━ 20:43:24 11s/step - accuracy: 0.8750 - loss: 0.3347

2025-04-12 21:34:27.615347: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:482] Shuffle buffer filled.


4224/6579 ━━━━━━━━━━━━━━━━━━━━ 3:15 83ms/step - accuracy: 0.9141 - loss: 0.2497

In [21]:
model.evaluate(test, verbose = 1)

NameError: name 'test' is not defined

In [ ]:
with open(f"models/model_history_{model_count + 1}.json", "w") as file:
    json.dump(model.history, file)

model.history

NameError: name 'model_count' is not defined

25/04/12 21:44:19 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /private/var/folders/7n/5wfxxtqx41zcc7qqdj_8rcl80000gn/T/blockmgr-aeb40553-acca-4960-9666-a31c7e2139f7. Falling back to Java IO way
java.io.IOException: Failed to delete: /private/var/folders/7n/5wfxxtqx41zcc7qqdj_8rcl80000gn/T/blockmgr-aeb40553-acca-4960-9666-a31c7e2139f7
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:166)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1(DiskBlockManager.scala:368)
	at org.apache.spark.storage.DiskBl

In [ ]:
# training_set = training_set.batch(batch_size = 32)
# training_set = training_set.prefetch(buffer_size = tf.data.AUTOTUNE)
# training_set = training_set.cache()
# training_set = training_set.shuffle(buffer_size = 1000, seed = 42)